In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.3 Circulant, Toeplitz, and the FFT as Diagonalization

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter VI — Structure, Graphs, and Fast Algorithms",
    number="6.3",
    title="Circulant, Toeplitz, and the FFT as Diagonalization",
    blurb="Every circulant matrix has the same eigenvectors — the Fourier "
    "modes — so the FFT diagonalizes all of them at once: convolution, "
    "solves, and deconvolution collapse from cubic to n log n, and the "
    "fast Fourier transform reveals itself as a change of basis that was "
    "computed cleverly.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

[§6.2](markov-perron-pagerank.ipynb) closed with eigenvalues on the unit
circle; this notebook is about the matrices whose eigen*vectors* live
there. A **circulant** matrix — each column the previous one rotated one
step — is completely described by its first column $c$, and the punchline
of the subject is that its eigendecomposition needs no eigensolver at
all: the eigenvectors are the Fourier modes for *every* circulant, and
the eigenvalues are `np.fft.fft(c)`, verified here as a matrix identity
to $10^{-13}$ and drawn as eigenvalues landing exactly on the symbol
curve. Everything fast follows: circular convolution becomes elementwise
multiplication in Fourier space, $Cx = b$ becomes three FFTs, and the
FLOP ledger — gated as arithmetic, per the course's rules — reads
$\tfrac23 n^3$ against $15\,n\log_2 n$: a factor of 229,000 at
$n = 8192$, of which the stopwatch (reported, never gated) shows 30,000.

**Toeplitz** matrices — constant along diagonals, no wraparound — lose
the exact diagonalization but keep the speed through two doors: the
**circulant embedding** (a $2n$ circulant whose top-left block *is* the
Toeplitz matrix, exactly — structural equality, and the FFT matvec that
follows), and Levinson recursion (`scipy.linalg.solve_toeplitz`,
$O(n^2)$, checked against the dense solve at $10^{-14}$). The finale is
the subject's signature application: a Gaussian blur as a circulant,
inverted by **regularized Fourier deconvolution** {cite}`hansen2010` —
the naive inverse filter divides by spectrally tiny numbers and returns
garbage (measured: overflow), while the Tikhonov version recovers a
smooth signal to 0.2%. A boxcar signal recovers only to ~6% at any
regularization — sharp edges live at all frequencies, and the blur
genuinely destroyed them: reported, because no algorithm can be gated
into un-losing information.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Strang {cite}`strang2023` on the FFT as matrix
> factorization; Golub and Van Loan {cite}`golub2013` Section 4.8 for
> Toeplitz systems; Hansen {cite}`hansen2010` for regularized
> deconvolution. The complex arithmetic conventions are
> [§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb)'s.

## Theory in brief

### One eigenvector basis for a whole algebra

The circulant with first column $c$ is
$C_{jk} = c_{(j - k) \bmod n}$ — shift structure baked into the index
arithmetic. Shifts commute with shifts, so all circulants share the
eigenvectors of the shift itself: the Fourier modes
$f_k(j) = e^{2\pi i jk/n}/\sqrt{n}$. In matrix form, with $F$ the unitary
DFT matrix and $\Lambda = \operatorname{diag}(\hat c)$,
$\hat c = \texttt{fft}(c)$:

```{math}
:label: eq-ct-diag
C\,F^{*} = F^{*}\Lambda
\qquad\Longleftrightarrow\qquad
C = F^{*}\,\operatorname{diag}(\texttt{fft}(c))\,F ,
```

one identity for every circulant at once. The eigenvalues trace the
**symbol** $f(\theta) = \sum_j c_j e^{-ij\theta}$ sampled at the $n$-th
roots of unity — a closed curve in the complex plane that the spectrum
sits on exactly.

### Convolution, and why the FFT owns it

Multiplying by $C$ *is* circular convolution: $(Cx)_j = \sum_k
c_{(j-k) \bmod n}\,x_k = (c \circledast x)_j$. Diagonalizing,

```{math}
:label: eq-ct-conv
c \circledast x \;=\; \texttt{ifft}\bigl(\texttt{fft}(c)\cdot
\texttt{fft}(x)\bigr),
\qquad
Cx = b \;\Longleftrightarrow\;
x = \texttt{ifft}\bigl(\texttt{fft}(b)/\texttt{fft}(c)\bigr),
```

each side three $O(n\log n)$ transforms against the direct $O(n^2)$ sum
or the $O(n^3)$ solve.

### Toeplitz: embedded, then solved

A Toeplitz matrix $T_{jk} = t_{j-k}$ has no wraparound, but the
$2n\times 2n$ circulant built from the column
$[t_0, \dots, t_{n-1}, 0, t_{-(n-1)}, \dots, t_{-1}]$ contains $T$ as
its top-left block *exactly*, so a Toeplitz matvec is a padded circulant
matvec at FFT speed. Solving keeps $O(n^2)$ through Levinson recursion
(the displacement structure of the matrix), and regularized
deconvolution handles the ill-posed case:

```{math}
:label: eq-ct-deconv
\hat x \;=\; \texttt{ifft}\!\left(
\frac{\overline{\hat k}\,\hat y}{|\hat k|^2 + \lambda}\right),
```

Tikhonov regularization ([§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb))
applied frequency by frequency — the whole SVD story of that notebook,
with the Fourier basis playing the singular vectors.

---
## Setup

Data only: the random first column and the seeded rng. The circulant
constructor — and the matrix it builds — belong to Exercise 1, where the
structure is the lesson.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_toeplitz, toeplitz

from ecp import validate
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps
N_C = 256



c_main = rng.standard_normal(N_C)

## Exercise 1: Diagonalized before any eigensolver runs

{eq}`eq-ct-diag` claims the eigendecomposition of every circulant is
known in advance. This exercise checks the claim three ways, none of
which involves ordering complex eigenvalues — the
[§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb) sorting hazard
designed out from the start.

**Part a)** Write `circulant(c)` — column $k$ is `np.roll(c, k)` — and
build `C_main` from the random first column, with `c_hat = np.fft.fft(c_main)`
beside it. Then build the unitary DFT matrix
$F_{jk} = e^{-2\pi i jk/n}/\sqrt{n}$ and gate the identity
$\lVert CF^{*} - F^{*}\Lambda\rVert / \lVert C\rVert_2 < 10^{-11}$ with
$\Lambda = \operatorname{diag}(\texttt{fft}(c))$ — the whole spectrum
checked in one matrix product, no eigensolver, no ordering.

**Part b)** Gate the eigenvector claim mode by mode: for each Fourier
mode $f_k$ (column of $F^{*}$),
$\lVert Cf_k - \hat c_k f_k\rVert < 10^{-11}\lVert C\rVert_2$.

**Part c)** Now bring the eigensolver and let it confirm: `np.linalg.eig`
returns the spectrum in *its own* order, so compare as multisets — every
computed eigenvalue within $10^{-11}$ of some entry of $\hat c$ and vice
versa (nearest-match in both directions, the order-free comparison).

**Part d)** Draw the circulant as a heatmap (the diagonal-stripe
structure), and the symbol picture for a circulant whose curve the eye
can actually follow: a *short-support* column (four harmonics), whose
symbol $f(\theta)$ is one smooth closed loop with the 256 eigenvalues
visibly on it — {eq}`eq-ct-diag` as a picture. (The random column's
symbol is a space-filling tangle; the identity holds just the same, but
a figure should be checkable by eye.) Gate the short-support version's
eigenvalues onto its symbol samples to $10^{-11}$ too.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.below(
    res_diag, 1e-11,
    "C F* = F* Lambda with Lambda = diag(fft(c)) (Eq. 1)",
    "the whole eigendecomposition as one matrix identity — no eigensolver, "
    "no ordering of complex eigenvalues, no 3.4 sorting hazard",
)
validate.below(
    mode_res, 1e-11,
    "and every Fourier mode is an eigenvector with eigenvalue fft(c)_k",
    "checked mode by mode against the matrix norm",
)
validate.check(
    match_fwd < 1e-11 and match_bwd < 1e-11 and match_short < 1e-11,
    "while np.linalg.eig confirms the spectrum as a multiset — for both "
    "circulants",
    f"nearest-match distances {match_fwd:.0e} / {match_bwd:.0e} for the "
    f"random column and {match_short:.0e} for the drawable four-harmonic "
    "one — order-free comparisons, because eig's ordering is the "
    "library's business",
)

## Exercise 2: Convolution three ways, priced by counting

{eq}`eq-ct-conv` says one operation wears three costumes.

**Part a)** Write `circular_convolve(a, b)` as the direct double loop —
$(a \circledast b)_j = \sum_k a_{(j-k)\bmod n} b_k$, exactly $n^2$
multiply-adds.

**Write this one yourself** — the two fast versions are checked against
it.

**Part b)** Gate the three-way agreement on random unit-scale vectors:
direct loop, circulant matvec `circulant(a) @ b`, and
`np.fft.ifft(np.fft.fft(a) * np.fft.fft(b)).real`, pairwise to
$10^{-12}$ (values of order $\sqrt{n}$ — the tolerance has two orders of
headroom over the observed $10^{-14}$).

**Part c)** Price them by counting, gated as arithmetic: the loop does
exactly $n^2 = 65{,}536$ multiply-adds; the FFT route costs about
$3 \cdot 5n\log_2 n \approx 30{,}720$ floating operations at $n = 256$ —
a modest $2\times$ here, but the *exponent* differs, so gate the model
ratio at $n = 8192$ instead: $n^2 / (15 n \log_2 n) = 42$. Report wall
times for both at $n = 256$; the stopwatch is testimony, not evidence
([§5.3](../05-numerical/sparse-matrices.ipynb)'s rule).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    gap_lm < 1e-12 and gap_lf < 1e-12,
    "direct loop, circulant matvec and FFT convolution agree (Eq. 2)",
    f"pairwise gaps {gap_lm:.0e}, {gap_lf:.0e} on unit-scale inputs: one "
    "operation, three costumes",
)
validate.check(
    model_ratio > 40.0,
    "and the FLOP model separates the exponents at n = 8192",
    f"n^2 against 15 n log2 n: {model_ratio:.0f}x by arithmetic — the "
    "stopwatch above testifies without being cross-examined (5.3's rule)",
)

## Exercise 3: The $O(n\log n)$ solve, and the honest stopwatch

Solving $Cx = b$ by {eq}`eq-ct-conv` is three FFTs and a division.

**Part a)** Gate correctness at $n = 256$:
`ifft(fft(b)/fft(c)).real` against `np.linalg.solve(C, b)` to
$10^{-11}$ (this circulant is well-conditioned — $\kappa = 79$ — so the
two backward-stable routes must agree at rounding level, per
[§5.1](../05-numerical/norms-conditioning-stability.ipynb)).

**Part b)** Scale up: at $n = 8192$, same gate — and the FLOP ledger,
gated as arithmetic: dense LU costs $\tfrac23 n^3 \approx 3.7\times
10^{11}$; the FFT solve about $15\,n\log_2 n \approx 1.6\times 10^{6}$ —
a ratio above $2\times10^{5}$. The manifest asked to gate "$\ge 100\times$
faster" on the clock; per the course's rules the clock is *reported*
(measured here: about $30{,}000\times$) and the model is gated.

**Part c)** Draw the runtime picture: FFT solve and dense solve against
$n$ on log axes, with the model slopes ($n\log n$ and $n^3$) as reference
lines — the measured points track the models' *slopes*, which is all a
stopwatch can honestly certify.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    gap_256 < 1e-11 and gap_big < 1e-11,
    "the three-FFT solve matches dense LU at both sizes (Eq. 2)",
    f"{gap_256:.0e} at n = 256, {gap_big:.0e} at n = 8192: two "
    "backward-stable routes on well-conditioned systems, priced by 5.1",
)
validate.check(
    flops_dense / flops_fft > 1e5,
    "at a gated FLOP ratio above one hundred thousand at n = 8192",
    f"{flops_dense/flops_fft:.0f}x by the model; the ~30,000x stopwatch is "
    "reported above — the manifest's wall-clock gate, converted per rule 2",
)

## Exercise 4: Toeplitz: embedded exactly, solved by displacement

No wraparound, two fast doors.

**Part a)** Build the SPD Toeplitz matrix $T$ with first column
$t_k = 2^{-k}$ (`scipy.linalg.toeplitz`), and its circulant embedding:
the $2n$-vector $[t_0, \dots, t_{n-1}, 0, t_{n-1}, \dots, t_1]$. Gate the
structural claim **exactly**: the top-left $n \times n$ block of the
embedding circulant equals $T$ entry for entry (`np.array_equal` — a
copy, not a computation).

**Part b)** Write `toeplitz_matvec_fft(t_col, x)`: pad $x$ with $n$
zeros, convolve with the embedding column by FFT, keep the first $n$
entries. Gate against the dense product to $10^{-13}$ — the arithmetic
rounds (unlike the block equality), but at unit scale it rounds at
$10^{-16}$.

**Part c)** Solve $Tx = b$ by Levinson recursion
(`scipy.linalg.solve_toeplitz`, $O(n^2)$ by displacement structure) and
gate against the dense solve to $10^{-10}$ — this matrix has
$\kappa = 9$, so the agreement should be, and is, at rounding level.
Report the operation-count ledger: Levinson $\sim 4n^2$, dense LU
$\tfrac23 n^3$ — the middle rung of the ladder $n^3 \to n^2 \to n\log n$
this notebook descends.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    block_exact,
    "the circulant embedding contains T as its top-left block, exactly",
    "a structural copy, not a computation — the legitimate exact gate, "
    "while the FFT matvec that uses it rounds like any arithmetic",
)
validate.below(
    gap_matvec, 1e-13,
    "the embedded FFT matvec reproduces the dense Toeplitz product",
    "pad, convolve, truncate: measured at rounding level on unit-scale "
    "input",
)
validate.below(
    gap_lev, 1e-10,
    "and Levinson recursion agrees with dense LU on the SPD system",
    "two backward-stable solvers at kappa = 9 — 5.1 prices the agreement, "
    "and O(n^2) displacement structure pays for it",
)

## Exercise 5: Deconvolution: the inverse problem in the Fourier basis

The subject's signature application, and [§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb)'s
regularization story with the FFT as the SVD.

**Part a)** Build a smooth signal (two Gaussians, widths 20 and 10) on
$n = 512$ samples, blur it with a circular Gaussian kernel of width 6,
and add noise at level $10^{-4}$. The blur's transfer function
$\hat k$ decays like $e^{-\theta^2}$: its analytic tail is $\sim10^{-77}$
at the Nyquist frequency, below anything the FFT's rounding can carry, so
the computed $\hat k$ bottoms out at rounding level and exact zeros —
information is not attenuated there, it is *gone*.

**Part b)** Confirm the naive inverse filter destroys itself: dividing
by $\hat k$ divides by rounding-level and exactly-zero entries. This run
returns an infinite relative error; an FFT that left $10^{-17}$ residue in
the tail would return a finite error near $10^{12}$ — the gate accepts
either, because only the destruction is mathematics. This is
[§5.1](../05-numerical/norms-conditioning-stability.ipynb)'s conditioning
story at its most extreme.

**Part c)** Gate the regularized version {eq}`eq-ct-deconv` at
$\lambda = 10^{-4}$: relative recovery error below the manifest's 5% —
measured 0.2%, because a smooth signal keeps essentially no energy where
the blur kills.

**Part d)** Now the honest failure: add a boxcar (sharp edges) to the
signal and sweep $\lambda$ over four decades — the error floors near 6%
at every $\lambda$ (reported, not gated: edges live at all frequencies,
the blur destroyed the high ones, and no choice of knob un-loses
information). Draw signal, blurred data, and both recoveries.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


```{admonition} With your assistant
:class: tip
The regularization parameter was stated, not chosen. Ask your assistant
for `lcurve(y, k_hat, lambdas)` returning residual norm and solution
norm per $\lambda$ — Hansen's L-curve — then check it against the
mathematics rather than a demo: (i) the residual norm is monotonically
increasing and the solution norm monotonically decreasing in $\lambda$
(both are exact consequences of Eq. 3, per frequency); (ii) the corner
of the curve on log axes sits within a decade of this notebook's
$10^{-4}$ for the smooth signal; (iii) at $\lambda \to 0$ the solution
norm explodes toward the naive filter's. The check is yours.
```

### Validation 5

In [ ]:
validate.check(
    rel_naive > 1e3 or not np.isfinite(rel_naive),
    "the naive inverse filter self-destructs (reported magnitude)",
    "dividing by a transfer function whose computed tail is rounding-level "
    "or exactly zero destroys the reconstruction — whether the wreck reads "
    "inf or 1e12 is the machine's; only the existence of the failure is gated",
)
validate.check(
    rel_reg < 0.05,
    "while Tikhonov deconvolution recovers the smooth signal within 5%",
    f"relative error {rel_reg:.4f} at lambda = 1e-4 and noise 1e-4 — "
    "2.4's regularization with the Fourier basis as the SVD (Eq. 3)",
)
validate.check(
    all(e > 0.02 for e in box_errs.values()),
    "and no regularization recovers the boxcar's edges (the honest floor)",
    f"errors {min(box_errs.values()):.3f}-{max(box_errs.values()):.3f} "
    "across four decades of lambda: the blur multiplied the edge "
    "frequencies by 1e-300 — gone is gone, which is the notebook's last "
    "and most important sentence about inverse problems",
)

---
## Notebook summary

**Every circulant is diagonalized before the eigensolver runs.**
$CF^{*} = F^{*}\operatorname{diag}(\texttt{fft}(c))$ held to $10^{-13}$
as a matrix identity, every Fourier mode passed its eigenvector residual,
and `np.linalg.eig`'s spectrum matched `fft(c)` as a multiset (nearest
match both directions — no complex sorting, by design). The eigenvalues
sat exactly on the symbol curve.

**One convolution, three costumes, one exponent.** The hand loop, the
circulant matvec and the FFT route agreed to $10^{-14}$; the FLOP ledger
— gated as arithmetic while stopwatches testified — read $n^2$ against
$15n\log_2 n$ for convolution and $\tfrac23 n^3$ against three FFTs for
the solve: a factor $2\times10^{5}$ at $n = 8192$, of which the clock
showed $30{,}000\times$ (0.3 ms against 9 s).

**Toeplitz keeps the speed without the diagonalization.** The $2n$
embedding contained $T$ *exactly* (structural copy, gated exactly; the
FFT matvec built on it rounds normally and matched at $10^{-15}$), and
Levinson's $O(n^2)$ recursion agreed with dense LU at rounding level on
the $\kappa = 9$ system — the middle rung of $n^3 \to n^2 \to n\log n$.

**Deconvolution is regularization in the Fourier basis.** The naive
inverse filter divided by $10^{-300}$ and overflowed; Tikhonov at
$\lambda = 10^{-4}$ recovered the smooth signal to **0.2%** (gated under
the manifest's 5%); and the boxcar variant floored near 6% across four
decades of $\lambda$ — reported, not gated, because the blur *destroyed*
the edges' frequencies and no knob un-loses information. [§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb)'s
entire SVD story, with Fourier modes as the singular vectors.

**Methods introduced.** `circulant` constructors, the DFT-matrix identity
check, symbol curves, order-free spectrum comparison, hand-rolled
circular convolution, FFT convolution and solves, FLOP-ledger gating with
reported stopwatches, circulant embedding of Toeplitz matvecs,
`scipy.linalg.solve_toeplitz`, and Tikhonov deconvolution with an
L-curve pointer.

## Outlook

- **Two dimensions is two kron factors.** Images blur by 2-D circulants
  — Kronecker products of 1-D ones — and
  [§6.4](kronecker-vec-separable.ipynb) makes that structure the whole
  story: `fft2` is `kron(F, F)` wearing speed.
- **The FFT itself is a matrix factorization.** Cooley–Tukey is
  $F_n = (\text{butterflies})(F_{n/2} \oplus F_{n/2})(\text{permutation})$
  — $\log n$ sparse factors, which is *why* it costs $n\log n$. Strang
  {cite}`strang2023` tells it as the most important factorization in
  applied mathematics.
- **Toeplitz asymptotics.** As $n \to \infty$ Toeplitz spectra distribute
  like their symbol's values (Szegő's theorem) — the circulant
  approximation this notebook used for a matvec becomes exact in the
  limit, which is the doorway to preconditioning Toeplitz systems with
  circulants ([§5.5](../05-numerical/krylov-gmres-preconditioning.ipynb)'s
  craft, specialized).
- **Convolution is everywhere in learning.** Convolutional layers are
  Toeplitz blocks with shared weights; the FFT view explains both their
  efficiency and their translation equivariance —
  [§8.3](../08-learning/linear-layer-backpropagation.ipynb) meets the
  layer as a matrix.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()